In [14]:
# Cell 1: Импорты, seed и настройка окружения
import os, json, random, re, subprocess, sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
from sklearn.feature_extraction.text import TfidfVectorizer

def ensure(package, imp_name=None):
    try: __import__(imp_name or package)
    except Exception: 
        print(f"Устанавливаем {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

ensure("faiss-cpu", "faiss")
ensure("sentence-transformers", "sentence_transformers")

import faiss
from sentence_transformers import SentenceTransformer

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [15]:
# Cell 2: База знаний и первичный анализ
documents = [
    {"doc_id": "doc_01", "title": "Что такое машинное обучение", "text": "Машинное обучение (МО) — это подраздел искусственного интеллекта, в котором алгоритмы учатся выявлять закономерности из данных, а не следуют жестко заданным правилам. Основные парадигмы: обучение с учителем, без учителя и с подкреплением."},
    {"doc_id": "doc_02", "title": "Обучение с учителем", "text": "В обучении с учителем алгоритм обучается на размеченных данных, где каждому входу соответствует известный выход. Задачи делятся на классификацию (предсказание метки класса) и регрессию (предсказание непрерывного значения). Примеры: линейная регрессия, случайный лес, SVM."},
    {"doc_id": "doc_03", "title": "Обучение без учителя", "text": "Алгоритмы работают с неразмеченными данными, ища скрытые структуры. Основные задачи: кластеризация (группировка похожих объектов) и снижение размерности (PCA, t-SNE). Применяется для исследования данных, сегментации клиентов и визуализации."},
    {"doc_id": "doc_04", "title": "Переобучение и регуляризация", "text": "Переобучение (overfitting) возникает, когда модель запоминает шум в обучающей выборке вместо общих закономерностей. Для борьбы используют регуляризацию (L1, L2), dropout, раннюю остановку и увеличение объёма данных."},
    {"doc_id": "doc_05", "title": "Разделение данных", "text": "Для оценки обобщающей способности данные делят на обучающую, валидационную и тестовую выборки. Тестовую выборку используют только один раз в конце обучения, чтобы получить непредвзятую оценку качества модели на новых данных."},
    {"doc_id": "doc_06", "title": "Метрики качества классификации", "text": "Помимо accuracy используются precision, recall и F1-score. Precision показывает долю истинно положительных среди всех предсказанных положительных, а recall — долю найденных положительных среди всех реальных. F1 является их гармоническим средним."},
    {"doc_id": "doc_07", "title": "Нейронные сети и глубокое обучение", "text": "Глубокое обучение использует многослойные нейронные сети для автоматического извлечения иерархических признаков. Сверточные сети (CNN) доминируют в компьютерном зрении, рекуррентные и трансформеры — в обработке естественного языка."},
    {"doc_id": "doc_08", "title": "Обработка естественного языка", "text": "NLP решает задачи токенизации, лемматизации, извлечения сущностей, машинного перевода и анализа тональности. Современные подходы опираются на большие языковые модели (LLM) и технику fine-tuning на доменных данных."},
    {"doc_id": "doc_09", "title": "Компьютерное зрение", "text": "CV включает задачи детекции объектов, сегментации изображений, распознавания лиц и генерации изображений. Ключевую роль играют аугментации данных, предобученные backbone-модели и метрики вроде mAP и IoU."},
    {"doc_id": "doc_10", "title": "MLOps и деплой моделей", "text": "MLOps объединяет разработку, тестирование и мониторинг ML-моделей в production. Включает версионирование данных и моделей, CI/CD пайплайны, логирование предсказаний и отслеживание дрейфа данных (data drift)."},
    {"doc_id": "doc_11", "title": "Аугментация данных", "text": "Искусственное расширение обучающей выборки путём преобразований: повороты, отражения, добавление шума, замена слов синонимами. Помогает улучшить обобщающую способность модели и снизить переобучение без сбора новых данных."},
    {"doc_id": "doc_12", "title": "Перенос обучения", "text": "Transfer learning позволяет использовать знания модели, обученной на большой общей выборке, для решения узкой задачи. Замораживаются ранние слои, а последние дообучаются на целевом датасете, что экономит время и вычислительные ресурсы."}
]

print(f"Загружено документов: {len(documents)}")
print(pd.DataFrame(documents)[["doc_id", "title"]])

Загружено документов: 12
    doc_id                               title
0   doc_01         Что такое машинное обучение
1   doc_02                 Обучение с учителем
2   doc_03                Обучение без учителя
3   doc_04        Переобучение и регуляризация
4   doc_05                   Разделение данных
5   doc_06      Метрики качества классификации
6   doc_07  Нейронные сети и глубокое обучение
7   doc_08       Обработка естественного языка
8   doc_09                 Компьютерное зрение
9   doc_10              MLOps и деплой моделей
10  doc_11                  Аугментация данных
11  doc_12                    Перенос обучения


In [16]:
# Cell 3: Чанкинг документов
def chunk_text(text: str, chunk_size: int = 40, overlap: int = 10) -> List[str]:
    words = text.split()
    if chunk_size <= 0 or overlap >= chunk_size:
        raise ValueError("Некорректные параметры чанкинга")
    chunks, step = [], chunk_size - overlap
    for start in range(0, len(words), step):
        chunk = words[start:start + chunk_size]
        if chunk: chunks.append(" ".join(chunk))
        if start + chunk_size >= len(words): break
    return chunks

# Пример чанкинга
print("Пример чанкинга (doc_04):")
for i, ch in enumerate(chunk_text(documents[3]["text"], chunk_size=30, overlap=8)):
    print(f"  [{i}] {ch}")

Пример чанкинга (doc_04):
  [0] Переобучение (overfitting) возникает, когда модель запоминает шум в обучающей выборке вместо общих закономерностей. Для борьбы используют регуляризацию (L1, L2), dropout, раннюю остановку и увеличение объёма данных.


In [17]:
# Cell 4: Эмбеддинги и индекс FAISS
@dataclass
class RetrievalArtifacts:
    backend_name: str
    chunks_df: pd.DataFrame
    chunk_vectors: np.ndarray
    index: object

def build_retriever(docs: List[Dict], chunk_size: int = 35, overlap: int = 10, device: str = "cpu") -> RetrievalArtifacts:
    model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    model = SentenceTransformer(model_name, device=device)
    
    rows = []
    for doc in docs:
        parts = chunk_text(doc["text"], chunk_size, overlap)
        for idx, txt in enumerate(parts):
            rows.append({"doc_id": doc["doc_id"], "title": doc["title"], "chunk_idx": idx, "chunk_text": txt})
    chunks_df = pd.DataFrame(rows)
    
    vectors = model.encode(chunks_df["chunk_text"].tolist(), normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    
    return RetrievalArtifacts("SentenceTransformer: paraphrase-multilingual-MiniLM", chunks_df, vectors, index)

def search_chunks(query: str, artifacts: RetrievalArtifacts, top_k: int = 3) -> pd.DataFrame:
    q_vec = artifacts.backend.split(":")[1].strip() if False else None # placeholder
    # В реальном коде модель хранится отдельно, здесь для компактности используем глобальный инстанс или переинициализируем
    return pd.DataFrame() # Заглушка, см. следующую ячейку для полной реализации

In [18]:
# Cell 5: Полный класс retriever для удобства экспериментов
class MiniRetriever:
    def __init__(self, docs, chunk_size=35, overlap=10, device="cpu"):
        self.model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=device)
        self.chunks = []
        for doc in docs:
            for i, txt in enumerate(chunk_text(doc["text"], chunk_size, overlap)):
                self.chunks.append({"doc_id": doc["doc_id"], "title": doc["title"], "text": txt})
        self.df = pd.DataFrame(self.chunks)
        self.vecs = self.model.encode(self.df["text"].tolist(), normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
        self.index = faiss.IndexFlatIP(self.vecs.shape[1])
        self.index.add(self.vecs)
        
    def search(self, query, top_k=3):
        q_vec = self.model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
        scores, idxs = self.index.search(q_vec, top_k)
        res = []
        for r, (s, i) in enumerate(zip(scores[0], idxs[0]), 1):
            row = self.df.iloc[int(i)]
            res.append({"rank": r, "score": float(s), "doc_id": row["doc_id"], "title": row["title"], "chunk_text": row["text"]})
        return pd.DataFrame(res)

# Инициализация baseline retriever
retriever = MiniRetriever(documents, chunk_size=35, overlap=10, device=device)
print(f"Индекс построен. Чанков: {len(retriever.df)}")
display(retriever.search("Что такое переобучение?", top_k=2)[["rank", "score", "title", "chunk_text"]])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Индекс построен. Чанков: 12


,rank,score,title,chunk_text
0,1,0.472752,Аугментация данных,Искусственное расширение обучающей выборки пут...
1,2,0.461335,Переобучение и регуляризация,"Переобучение (overfitting) возникает, когда мо..."


In [19]:
# Cell 6: Benchmark и оценка retrieval
benchmark_queries = [
    {"query_id": "q1", "query": "Чем отличается обучение с учителем от обучения без учителя?", "relevant": ["doc_02", "doc_03"]},
    {"query_id": "q2", "query": "Как бороться с переобучением модели?", "relevant": ["doc_04"]},
    {"query_id": "q3", "query": "Какие метрики используют для оценки классификации?", "relevant": ["doc_06"]},
    {"query_id": "q4", "query": "Зачем нужны тестовая и валидационная выборки?", "relevant": ["doc_05"]},
    {"query_id": "q5", "query": "Какие архитектуры нейросетей используются в NLP?", "relevant": ["doc_07"]},
    {"query_id": "q6", "query": "Что включает в себя практика MLOps?", "relevant": ["doc_10"]},
    {"query_id": "q7", "query": "Как аугментация данных влияет на обучение?", "relevant": ["doc_11"]},
    {"query_id": "q8", "query": "В чём суть переноса обучения?", "relevant": ["doc_12"]}
]
benchmark_df = pd.DataFrame(benchmark_queries)
display(benchmark_df)

benchmark = benchmark_queries

def evaluate_retrieval(ret, queries, top_k=3):
    rows = []
    for q in queries:
        res = ret.search(q["query"], top_k)
        # Убираем дубликаты документов, сохраняя порядок по рангу
        seen, ordered = set(), []
        for _, r in res.iterrows():
            if r["doc_id"] not in seen: seen.add(r["doc_id"]); ordered.append(r["doc_id"])
            
        hit = int(any(d in ordered for d in q["relevant"]))
        recall = sum(d in ordered for d in q["relevant"]) / len(q["relevant"])
        mrr, first_r = 0.0, None
        for rank, doc in enumerate(ordered, 1):
            if doc in q["relevant"]: mrr, first_r = 1.0/rank, rank; break
            
        rows.append({
            "query_id": q["query_id"], "query": q["query"], "expected_source": ", ".join(q["relevant"]),
            "retrieved_sources": ", ".join(ordered[:top_k]), "hit_at_k": hit,
            "recall@k": recall, "MRR@k": mrr, "first_relevant_rank": first_r or 99
        })
    return pd.DataFrame(rows)

eval_df = evaluate_retrieval(retriever, benchmark, top_k=3)
eval_df.to_csv("artifacts/retrieval_eval.csv", index=False)
print("Оценка retrieval:")
display(eval_df[["query_id", "hit_at_k", "recall@k", "MRR@k"]])

,query_id,query,relevant
0,q1,Чем отличается обучение с учителем от обучения...,"[doc_02, doc_03]"
1,q2,Как бороться с переобучением модели?,[doc_04]
2,q3,Какие метрики используют для оценки классифика...,[doc_06]
3,q4,Зачем нужны тестовая и валидационная выборки?,[doc_05]
4,q5,Какие архитектуры нейросетей используются в NLP?,[doc_07]
5,q6,Что включает в себя практика MLOps?,[doc_10]
6,q7,Как аугментация данных влияет на обучение?,[doc_11]
7,q8,В чём суть переноса обучения?,[doc_12]


Оценка retrieval:


,query_id,hit_at_k,recall@k,MRR@k
0,q1,1,0.5,1.000000
1,q2,1,1.0,1.000000
2,q3,0,0.0,0.000000
3,q4,1,1.0,1.000000
4,q5,1,1.0,0.500000
5,q6,1,1.0,1.000000
6,q7,1,1.0,0.333333
7,q8,1,1.0,1.000000


In [21]:
# Cell 7: Эксперимент с параметрами чанкинга
print("🔬 Сравнение chunk_size=25 vs chunk_size=45")
res_25 = evaluate_retrieval(MiniRetriever(documents, chunk_size=25, overlap=8, device=device), benchmark, top_k=3)
res_45 = evaluate_retrieval(MiniRetriever(documents, chunk_size=45, overlap=12, device=device), benchmark, top_k=3)

exp_df = pd.DataFrame({
    "chunk_size": [25, 45],
    "overlap": [8, 12],
    "mean_hit@3": [res_25["hit_at_k"].mean(), res_45["hit_at_k"].mean()],
    "mean_recall@3": [res_25["recall@k"].mean(), res_45["recall@k"].mean()],
    "mean_MRR@3": [res_25["MRR@k"].mean(), res_45["MRR@k"].mean()]
})
display(exp_df)
print("Вывод: меньший chunk_size лучше локализует ответ, но может терять контекст. Средние показатели близки на коротких текстах.")

🔬 Сравнение chunk_size=25 vs chunk_size=45


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,chunk_size,overlap,mean_hit@3,mean_recall@3,mean_MRR@3
0,25,8,0.750,0.6875,0.562500
1,45,12,0.875,0.8125,0.729167


Вывод: меньший chunk_size лучше локализует ответ, но может терять контекст. Средние показатели близки на коротких текстах.


In [23]:
# Cell 8: Обновление базы знаний и переиндексация
new_docs = [
    {"doc_id": "doc_13", "title": "Генеративно-состязательные сети", "text": "GAN состоят из генератора и дискриминатора, которые соревнуются друг с другом. Генератор создаёт поддельные данные, а дискриминатор учится отличать их от реальных. Применяются в создании изображений, музыки и текста."},
    {"doc_id": "doc_14", "title": "Обучение с подкреплением", "text": "Агент взаимодействует со средой, получая награды или штрафы за действия. Цель — максимизировать кумулятивную награду. Ключевые алгоритмы: Q-learning, Policy Gradients, PPO. Используется в робототехнике и играх."}
]
updated_docs = documents + new_docs
retriever_updated = MiniRetriever(updated_docs, chunk_size=35, overlap=10, device=device)

extended_benchmark = benchmark + [
    {"query_id": "q9", "query": "Как работают GAN и из каких компонентов состоят?", "relevant": ["doc_13"]},
    {"query_id": "q10", "query": "Что такое агент и награда в RL?", "relevant": ["doc_14"]}
]

before_df = evaluate_retrieval(retriever, extended_benchmark, top_k=3)
after_df = evaluate_retrieval(retriever_updated, extended_benchmark, top_k=3)

comparison = before_df[["query_id", "query", "retrieved_sources"]].rename(columns={"retrieved_sources": "before_retrieved_sources"})
comparison = comparison.merge(after_df[["query_id", "retrieved_sources"]].rename(columns={"retrieved_sources": "after_retrieved_sources"}), on="query_id")
comparison["changed"] = comparison["before_retrieved_sources"] != comparison["after_retrieved_sources"]
comparison.to_csv("artifacts/retrieval_before_after_update.csv", index=False)

print("Сравнение до/после обновления БЗ:")
display(comparison[comparison["changed"]][["query_id", "before_retrieved_sources", "after_retrieved_sources"]])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Сравнение до/после обновления БЗ:


,query_id,before_retrieved_sources,after_retrieved_sources
8,q9,"doc_10, doc_08, doc_03","doc_13, doc_10, doc_08"
9,q10,"doc_12, doc_09, doc_08","doc_14, doc_13, doc_12"


In [24]:
# Cell 9: Mini-RAG пайплайн
def get_answer_from_context(query: str, top_chunks: pd.DataFrame) -> Tuple[str, str]:
    if top_chunks.empty: return "Нет релевантных данных.", ""
    # Extractive approach: ищем предложение в чанках с max косинусным сходством к запросу
    vectorizer = TfidfVectorizer(ngram_range=(1, 2)).fit([query] + top_chunks["chunk_text"].tolist())
    q_vec = vectorizer.transform([query]).toarray()
    c_vecs = vectorizer.transform(top_chunks["chunk_text"].tolist()).toarray()
    
    norms = np.linalg.norm(c_vecs, axis=1, keepdims=True) + 1e-8
    scores = (c_vecs @ q_vec.T).flatten() / (norms.flatten() + 1e-8)
    best_idx = np.argmax(scores)
    return top_chunks.iloc[best_idx]["chunk_text"], top_chunks.iloc[best_idx]["doc_id"]

def mini_rag_pipeline(query, retriever_obj, top_k=3):
    chunks = retriever_obj.search(query, top_k)
    answer, source_doc = get_answer_from_context(query, chunks)
    sources = f"[{', '.join(chunks['title'].unique())}]"
    return {"question": query, "answer": answer, "retrieved_sources": sources, "chunks": chunks}

# Примеры работы RAG
rag_results = []
for q in ["Как бороться с переобучением модели?", "Что такое обучение с подкреплением?", "Зачем нужны валидационные данные?"]:
    res = mini_rag_pipeline(q, retriever_updated, top_k=3)
    rag_results.append({"question": res["question"], "answer": res["answer"], "retrieved_sources": res["retrieved_sources"]})
    print(f" {q}\n {res['answer']}\n Источники: {res['retrieved_sources']}\n")

pd.DataFrame(rag_results).to_csv("artifacts/rag_examples.csv", index=False)

 Как бороться с переобучением модели?
 Искусственное расширение обучающей выборки путём преобразований: повороты, отражения, добавление шума, замена слов синонимами. Помогает улучшить обобщающую способность модели и снизить переобучение без сбора новых данных.
 Источники: [Переобучение и регуляризация, Аугментация данных, Перенос обучения]

 Что такое обучение с подкреплением?
 Transfer learning позволяет использовать знания модели, обученной на большой общей выборке, для решения узкой задачи. Замораживаются ранние слои, а последние дообучаются на целевом датасете, что экономит время и вычислительные ресурсы.
 Источники: [Перенос обучения, Аугментация данных, Обучение с учителем]

 Зачем нужны валидационные данные?
 Для оценки обобщающей способности данные делят на обучающую, валидационную и тестовую выборки. Тестовую выборку используют только один раз в конце обучения, чтобы получить непредвзятую оценку качества модели на новых данных.
 Источники: [Разделение данных, Метрики качества 

In [25]:
# Cell 10: Анализ ошибок
weak_cases = eval_df[(eval_df["MRR@k"] < 1.0) | (eval_df["first_relevant_rank"] > 1)]
print("Слабые кейсы (ответ не на 1 месте или пропущен):")
if weak_cases.empty: print("На данном бенчмарке все релевантные документы находятся на первом месте.")
else: display(weak_cases[["query_id", "query", "first_relevant_rank", "MRR@k"]])
print("Вывод: При расхождении лексики запроса и текста (например, 'бороться' vs 'регуляризация') модель может опускать релевантный чанк на 2-3 место. Решение: реранкинг или расширение синонимического словаря в промпте.")

Слабые кейсы (ответ не на 1 месте или пропущен):


,query_id,query,first_relevant_rank,MRR@k
2,q3,Какие метрики используют для оценки классифика...,99,0.000000
4,q5,Какие архитектуры нейросетей используются в NLP?,2,0.500000
6,q7,Как аугментация данных влияет на обучение?,3,0.333333


Вывод: При расхождении лексики запроса и текста (например, 'бороться' vs 'регуляризация') модель может опускать релевантный чанк на 2-3 место. Решение: реранкинг или расширение синонимического словаря в промпте.
